# Lista 03 — Visualização e Análise Exploratória

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

Referente ao módulo **03 · Visualização e EDA**

---


Esta lista fecha o conteúdo técnico da capacitação. Ela cobre Matplotlib, Seaborn,
estatística descritiva e — no exercício final — o ciclo completo de análise exploratória.

Os gráficos serão avaliados por você mesmo com um critério simples: **alguém que não viu
seus dados entenderia o que ele mostra em dez segundos?**


**11 exercícios** (3 de teoria, 8 de código) ·
**Tempo estimado:** 2h30

> **Atenção — Use este arquivo depois de tentar.** As soluções aqui são uma entre várias
> possíveis — se a sua for diferente e funcionar, ela também está certa. O que importa
> é você conseguir explicar cada linha do que escreveu.

Cada exercício traz a solução comentada e, quando cabe, uma observação sobre o porquê
daquela escolha.

## Preparação

### Antes de começar — se você está no Google Colab

Este notebook lê arquivos da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "05_Exercicios"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", palette="deep")

acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
empresas = pd.read_csv("../data/empresas_b3.csv")
indicadores = pd.read_csv("../data/indicadores_macro.csv", parse_dates=["data"])

acoes = acoes.merge(empresas, on="ticker", how="left", validate="m:1")
acoes = acoes.sort_values(["ticker", "data"])
acoes["retorno_diario"] = acoes.groupby("ticker")["fechamento_ajustado"].pct_change() * 100
acoes["ano"] = acoes["data"].dt.year

print("Base pronta:", acoes.shape)
acoes.head(3)

---

## Exercício 1 (teoria) — Escolhendo o gráfico


Para cada pergunta, diga qual tipo de gráfico você usaria e o que ficaria em cada eixo:

1. Como o preço da PETR4 evoluiu nos últimos cinco anos?
2. Qual setor teve o maior retorno médio?
3. Os retornos diários da VALE3 se concentram em torno de que valor?
4. Ações mais voláteis renderam mais?
5. Quais ativos se movimentam juntos?

**Resposta:**


1. **Linha** — data no eixo X, preço no eixo Y. A linha sugere continuidade, adequada a
   séries temporais.
2. **Barras** (de preferência horizontais e ordenadas) — retorno médio no eixo do
   comprimento, setor no outro.
3. **Histograma** — retorno diário no eixo X, contagem de pregões no eixo Y. Um
   **boxplot** também responde, de forma mais resumida.
4. **Dispersão** — volatilidade no eixo X, retorno no eixo Y, um ponto por ativo.
5. **Mapa de calor** da matriz de correlação — ativos nos dois eixos, correlação na cor.

---

## Exercício 2 (código) — Um gráfico bem-feito


Faça um gráfico de linha da evolução do `fechamento_ajustado` da **ITUB4** ao longo de
todo o período.

Requisitos: título descritivo, rótulos nos dois eixos **com unidade**, grade discreta e
tamanho de figura adequado.

In [ ]:
itub = acoes[acoes["ticker"] == "ITUB4"].sort_values("data")

fig, ax = plt.subplots(figsize=(11, 4.5))

ax.plot(itub["data"], itub["fechamento_ajustado"], color="#1f4e79", linewidth=1.3)

ax.set_title("ITUB4 — preço de fechamento ajustado (2021–2025)", fontsize=13, pad=12)
ax.set_xlabel("Data")
ax.set_ylabel("Preço ajustado (R$)")
ax.grid(alpha=0.3)

plt.show()

A diferença entre este gráfico e um `plt.plot(x, y)` solto é inteira nos rótulos. Um
gráfico sem título e sem unidade não pode ser colado em lugar nenhum — nem num relatório,
nem numa mensagem para o time.

---

## Exercício 3 (código) — Comparação justa


Compare a evolução de **PETR4, ITUB4 e MGLU3** em um único gráfico.

Como os preços têm níveis muito diferentes, normalize cada série para **base 100** no
primeiro pregão. Inclua legenda e uma linha horizontal de referência em 100.

Escreva, em um comentário, o que o gráfico mostra.


> **Dica:** Para cada ativo: `serie / serie.iloc[0] * 100`.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for ticker in ["PETR4", "ITUB4", "MGLU3"]:
    serie = acoes[acoes["ticker"] == ticker].sort_values("data")
    base = serie["fechamento_ajustado"].iloc[0]
    ax.plot(serie["data"], serie["fechamento_ajustado"] / base * 100,
            label=ticker, linewidth=1.4)

ax.axhline(100, color="gray", linestyle="--", linewidth=1)
ax.set_title("Retorno acumulado — base 100 no primeiro pregão de 2021", fontsize=13, pad=12)
ax.set_xlabel("Data")
ax.set_ylabel("Índice (100 = início do período)")
ax.legend()
ax.grid(alpha=0.3)

plt.show()

# PETR4 multiplicou o capital investido; ITUB4 teve alta moderada; MGLU3 perdeu
# quase todo o valor. A dispersão entre ativos do mesmo mercado é enorme.

A normalização por base 100 é o que torna a comparação **justa**. Sem ela, o gráfico
compara reais com reais em ativos de níveis de preço diferentes — e o ativo mais caro
parece "melhor" só por ocupar a parte de cima do gráfico.

---

## Exercício 4 (código) — Distribuição dos retornos


Faça um histograma dos retornos diários da **VALE3**, com uma linha vertical na média.

Depois responda, com código: qual porcentagem dos pregões teve retorno entre −2% e +2%?

In [ ]:
vale = acoes.loc[acoes["ticker"] == "VALE3", "retorno_diario"].dropna()

fig, ax = plt.subplots(figsize=(9, 4.5))

sns.histplot(vale, bins=60, ax=ax, color="#1f4e79")
ax.axvline(vale.mean(), color="#c0392b", linewidth=2,
           label=f"média = {vale.mean():.3f}%")

ax.set_title("Distribuição dos retornos diários da VALE3 (2021–2025)")
ax.set_xlabel("Retorno diário (%)")
ax.set_ylabel("Número de pregões")
ax.legend()

plt.show()

dentro = vale.between(-2, 2).mean()
print(f"{dentro:.1%} dos pregões tiveram retorno entre -2% e +2%")

A distribuição é aproximadamente simétrica e concentrada perto de zero, mas com **caudas
mais grossas** que uma normal: os dias de variação extrema são raros e, ainda assim, mais
frequentes do que a curva do sino preveria. É por isso que modelos de risco baseados em
normalidade subestimam sistematicamente a chance de perdas grandes.

---

## Exercício 5 (código) — Comparando grupos com boxplot


Faça um boxplot dos retornos diários **por ativo**, ordenando os ativos pela
volatilidade (do menos para o mais volátil).

Acrescente uma linha horizontal em zero. Em seguida, imprima a tabela de volatilidade que
justifica a ordem escolhida.


> **Dica:** `order=` recebe a lista de categorias na ordem desejada.

In [ ]:
volatilidade = acoes.groupby("ticker")["retorno_diario"].std().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(data=acoes, x="ticker", y="retorno_diario", order=volatilidade.index, ax=ax)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Retornos diários por ativo, ordenados por volatilidade")
ax.set_xlabel("")
ax.set_ylabel("Retorno diário (%)")

plt.show()

print("Volatilidade diária (desvio padrão, %):")
print(volatilidade.round(3))

Ordenar transforma um amontoado de caixas em um **ranking**. É uma das intervenções de
maior retorno em visualização: custa uma linha e muda completamente a legibilidade.

---

## Exercício 6 (teoria) — Média ou mediana?


Em uma base de clientes, o patrimônio investido tem média de R$ 123 mil e mediana de
R$ 84 mil.

1. O que essa diferença diz sobre o formato da distribuição?
2. Qual das duas medidas você usaria para descrever "o cliente típico"? Por quê?
3. Existe alguma situação em que a média seria a medida mais apropriada aqui?

**Resposta:**


1. A média ser bem maior que a mediana indica **assimetria à direita**: a maioria dos
   clientes tem patrimônio baixo, e uma minoria com valores muito altos puxa a média para
   cima. É o formato típico de qualquer variável de dinheiro.

2. A **mediana**. Ela responde "metade dos clientes tem menos que isso", que é a definição
   operacional de típico. Dizer que o cliente médio tem R$ 123 mil descreve alguém que
   quase não existe na base.

3. Sim. A média é a medida certa quando o que interessa é o **total**: receita esperada,
   custódia total sob gestão, tamanho de mercado. Média × número de clientes dá o total;
   mediana × número de clientes não dá nada. As duas medidas respondem perguntas
   diferentes — o erro é usar uma para responder a pergunta da outra.

---

## Exercício 7 (código) — Estatísticas descritivas


Para cada ativo, monte uma tabela com: média, mediana, desvio padrão, assimetria,
percentil 5 (o VaR diário de 95%) e o pior retorno do período — todos sobre o retorno
diário.

Ordene pelo percentil 5, do pior para o melhor, e comente o que a coluna de assimetria
mostra.


> **Dica:** `.agg()` aceita funções nomeadas; para o percentil, use `lambda serie: serie.quantile(0.05)`.

In [ ]:
estatisticas = acoes.groupby("ticker")["retorno_diario"].agg(
    media="mean",
    mediana="median",
    desvio="std",
    assimetria="skew",
    var_95=lambda serie: serie.quantile(0.05),
    pior_dia="min",
).round(3)

estatisticas.sort_values("var_95")

Duas leituras:

- o **VaR 95%** diz o limite das perdas em 95% dos dias — e, por construção, nada diz
  sobre o que acontece nos 5% restantes. Compare a coluna `var_95` com a coluna
  `pior_dia` e veja o tamanho do que a medida deixa de fora;
- a **assimetria** negativa na maioria dos ativos significa cauda esquerda mais longa: as
  quedas extremas são maiores que as altas extremas. É uma característica conhecida de
  mercados de ações, e uma má notícia para quem está investido.

---

## Exercício 8 (código) — Correlação e mapa de calor


1. monte uma tabela com uma coluna por ativo e uma linha por data, contendo os retornos
   diários (use `pivot_table`);
2. calcule a matriz de correlação;
3. desenhe o mapa de calor, com os valores anotados, escala divergente centrada em zero e
   limites fixados em −1 e +1;
4. imprima o par de ativos **mais** correlacionado e o **menos** correlacionado.


> **Dica:** Para o item 4, use `.unstack()` na matriz e descarte os pares de um ativo com ele mesmo.

In [ ]:
retornos = acoes.pivot_table(index="data", columns="ticker", values="retorno_diario")
correlacao = retornos.corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(correlacao, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlação entre os retornos diários", pad=14)
ax.set_xlabel("")
ax.set_ylabel("")
plt.show()

pares = correlacao.unstack()
pares = pares[pares < 0.999]          # descarta a diagonal (ativo com ele mesmo)

print("Par mais correlacionado :", pares.idxmax(), f"({pares.max():.3f})")
print("Par menos correlacionado:", pares.idxmin(), f"({pares.min():.3f})")

A escala divergente centrada em zero (`cmap="RdBu_r"`, `center=0`) é a escolha correta
aqui porque a correlação tem um ponto neutro natural. Fixar `vmin=-1, vmax=1` garante que
a mesma cor signifique o mesmo valor em qualquer versão do gráfico — sem isso, o pandas
ajusta a escala aos dados e duas versões do mesmo mapa ficam incomparáveis.

---

## Exercício 9 (teoria) — Correlação e causalidade


Um colega calcula a correlação entre o retorno mensal de uma ação de varejo e a Selic, e
encontra −0,45. Ele conclui: *"provamos que a alta de juros derrubou a ação"*.

1. Aponte pelo menos **três** problemas com essa conclusão.
2. O que ele poderia fazer para investigar a hipótese de forma mais sólida?

**Resposta:**


**1. Problemas:**

- **Correlação não é causalidade.** Uma terceira variável pode causar as duas coisas: o
  mesmo cenário macroeconômico que levou o Banco Central a subir juros (inflação alta,
  atividade fraca) também afeta diretamente as vendas do varejo.
- **A direção da causalidade não está estabelecida** pelo número. Ele mostra
  co-movimento, não quem move quem.
- **Tamanho da amostra.** Se são poucos meses, −0,45 é compatível com acaso. Um
  coeficiente sem o tamanho da amostra ao lado não é interpretável.
- **Uma ação só.** Não há como saber se o resultado é sobre "varejo e juros" ou sobre
  aquela empresa específica, que pode ter tido problemas próprios no período.
- **Nível × surpresa.** Preços reagem a informação nova. Correlacionar com o *nível* da
  taxa, que o mercado já conhece, mede a coisa errada.

**2. Como investigar melhor:**

- usar **muitas empresas** do setor, e comparar com um grupo de controle de outro setor;
- usar a **variação não antecipada** da taxa (surpresa em relação ao esperado), em vez do
  nível;
- **controlar** por outras variáveis (câmbio, atividade, endividamento da empresa);
- ampliar o **período**, cobrindo mais de um ciclo de juros;
- e, acima de tudo, formular o **mecanismo econômico** antes de olhar o número — e
  verificar se as implicações desse mecanismo também aparecem nos dados.

---

## Exercício 10 (código) — Mapa risco × retorno


Monte o gráfico clássico de risco × retorno:

- eixo X: volatilidade anualizada (desvio padrão diário × √252);
- eixo Y: retorno anualizado (média diária × 252);
- um ponto por ativo, com o ticker escrito ao lado;
- linha horizontal em zero.

Depois imprima a tabela ordenada por retorno dividido por risco.

In [ ]:
mapa = acoes.groupby("ticker")["retorno_diario"].agg(["mean", "std"])
mapa["retorno_anual"] = mapa["mean"] * 252
mapa["risco_anual"] = mapa["std"] * np.sqrt(252)
mapa["retorno_por_risco"] = mapa["retorno_anual"] / mapa["risco_anual"]

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(mapa["risco_anual"], mapa["retorno_anual"], s=140, color="#1f4e79", zorder=3)
for ticker, linha in mapa.iterrows():
    ax.annotate(ticker, (linha["risco_anual"], linha["retorno_anual"]),
                xytext=(9, 5), textcoords="offset points", fontsize=11)

ax.axhline(0, color="black", linewidth=0.9)
ax.set_title("Risco × retorno anualizados (2021–2025)", fontsize=13, pad=12)
ax.set_xlabel("Volatilidade anualizada (%)")
ax.set_ylabel("Retorno anualizado (%)")
ax.grid(alpha=0.3)

plt.show()

print(mapa[["retorno_anual", "risco_anual", "retorno_por_risco"]]
      .round(2).sort_values("retorno_por_risco", ascending=False))

Quanto mais **acima** e mais à **esquerda**, melhor. A última coluna é a ideia por trás do
índice de Sharpe: retorno por unidade de risco.

E vale repetir o alerta: tudo isso é **descritivo**. Descreve 2021–2025 e não autoriza
nenhuma afirmação sobre o futuro.

---

## Exercício 11 (código) — Desafio — um ciclo completo de EDA


Escolha **uma** das perguntas abaixo (ou formule a sua) e percorra o ciclo completo:
pergunta → exploração → descoberta → hipótese.

- Os retornos são diferentes conforme o dia da semana?
- O volume negociado aumenta nos dias de queda forte?
- Existe relação entre a amplitude diária (máxima − mínima) e o retorno do dia?
- Algum ativo se comporta de forma diferente dos demais em meses de IPCA alto?

Requisitos:

1. escreva a **pergunta** em uma célula de texto, antes de olhar qualquer resultado;
2. produza pelo menos **um resumo numérico** e **um gráfico**;
3. escreva a **descoberta** em duas ou três frases, com números;
4. escreva uma **hipótese** explicativa e diga **o que você não pode afirmar** com esses
   dados.


> **Dica:** `acoes["data"].dt.day_name()` dá o dia da semana. Para o item 4, sempre pense: tamanho da amostra, causalidade, período específico.

In [ ]:
# ============================================================
# PERGUNTA
# O volume negociado é maior nos dias de queda forte do que nos demais dias?
# ============================================================

acoes["dia_de_queda_forte"] = acoes["retorno_diario"] < -3

# EXPLORAÇÃO — resumo numérico
resumo = acoes.groupby("dia_de_queda_forte").agg(
    pregoes=("volume", "count"),
    volume_mediano=("volume", "median"),
    volume_medio=("volume", "mean"),
).round(0)

print(resumo)

razao = (resumo.loc[True, "volume_mediano"] / resumo.loc[False, "volume_mediano"])
print(f"\nVolume mediano em quedas fortes é {razao:.2f}× o dos demais dias")

# EXPLORAÇÃO — gráfico
fig, ax = plt.subplots(figsize=(9, 5))

sns.boxplot(data=acoes.dropna(subset=["retorno_diario"]),
            x="dia_de_queda_forte", y="volume", showfliers=False, ax=ax)

ax.set_title("Volume negociado em dias de queda forte (< -3%) e nos demais")
ax.set_xlabel("Dia de queda forte")
ax.set_ylabel("Volume (número de ações)")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Não", "Sim"])

plt.show()

# Conferindo se o padrão vale para todos os ativos, e não só na média geral
por_ativo = (
    acoes.groupby(["ticker", "dia_de_queda_forte"])["volume"].median()
    .unstack()
)
por_ativo["razao"] = (por_ativo[True] / por_ativo[False]).round(2)
print(por_ativo.round(0))

Uma redação possível para as etapas 3 e 4:

> **DESCOBERTA.** O volume mediano nos dias de queda superior a 3% é sensivelmente
> maior que nos demais pregões, e o padrão se repete em todos os ativos da amostra — não
> é efeito de um único papel puxando a média. O boxplot mostra que a distribuição inteira
> se desloca para cima, não apenas alguns dias extremos.
>
> **HIPÓTESE.** Quedas fortes coincidem com a chegada de informação relevante
> (resultados, notícias, dados macro), e informação nova gera negociação: quem discorda do
> novo preço compra, quem se assusta vende. Some-se a isso a execução de ordens de stop,
> que se acumulam em movimentos bruscos.
>
> **ATENÇÃO — O QUE NÃO POSSO AFIRMAR.** (1) Nada sobre direção de causalidade — volume
> alto e queda forte podem ser dois efeitos da mesma causa, e não um causando o outro. (2)
> O recorte de −3% é arbitrário; convém verificar se o resultado sobrevive a outros
> limites. (3) A amostra tem 8 ações grandes e líquidas em um período específico; não vale
> para "a bolsa". (4) Não distingui quedas do ativo de quedas do mercado inteiro — o
> mecanismo pode ser bem diferente nos dois casos.

Repare que a solução **testa a robustez** (o padrão vale por ativo?) antes de concluir, e
que a lista do que não se pode afirmar é tão longa quanto a descoberta. É assim que se
escreve uma EDA honesta.

---

## Fim do gabarito

Se você resolveu a maior parte sem consultar, pode seguir para o próximo módulo. Se
consultou muito, vale refazer os exercícios em que travou — desta vez, sem olhar.